# Markov Chain Notebook

This notebook will give elaborate explanation of markov chains, how they work, and also use distribution matrix, prove formulas and experiment with markov chain. Finally we will run an experiment where we will construct communicating classes and experiment markov chain on already exisiting applications

In [ ]:
# All library imports go here
import os
import sys 
import random
import re

from collections import defaultdict, Counter
from pathlib import Path
from functools import cache
from typing import Literal,Tuple

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "information_theory" / "udhr_eng.txt"

# Increase recursion limit
sys.setrecursionlimit(5000)

## Markov Chain

Markov Chain is a stochastic system where the transition probability from one state to another is i]dependent of the path it took to reach the starting location. It's main assumption is $ P(X_{n+1} =j | X_n=i) = P(X_{n+1} =j | X_n=i, X_n-1,...,X_0) $.

Based on this we can calculate the probability for reaching a certain position after some time using recursion. $ P(X_{n+1} =j | X_n=i) = \sum_k^n P(X_{n} =k | X_n=i)P_{kj} $

In [ ]:
graph = [
    [0.5,0.5],
    [.2,.8]
]

# Calculate the probability that we will be at 1 or 2 after 100,1000 steps 

@cache
def rec(n,start,end):
    global graph 
    if n==1:
        return graph[start][end]

    sol = 0

    for j in range(len(graph[start])):
        sol += rec(n-1,start,j) * graph[j][end]

    return sol

print(rec(2,0,1))
print(rec(50,0,1))
print(rec(100,0,1)) # Converges to 5/7
print(rec(100,0,0)) # Converges to 2/7

print(rec(100,0,0)==rec(100,1,0))

**Points to remember**

1. When a markov chain converges it will give us the probability that we will end up at a certain point regradless of the starting position.
2. Not all markov chains converge. Some could be periodic.
3. The other case where this fails is when some portion of the chain is unreachable and as a result the final probability is dependent on the initial starting position

States can be **transient** or **recurrent**. Recurrent states are states where you can go from state a to any other possible state and there is a way to return back to a. A state that is not recurrent is transient. If state is transient then $P(X_i)=0$. The 3rd condition specifically fails when there are more than one recurrent class. **Recurrent Class**: are states that communicate with each other.

Let's evaluate the chain of events that Andrey Markov has followed to disprove that law of large numbers doesn't imply independent actions. Markov used Puskin's poem Eugene Onegin and first calculated the percentage of vowels and consonants appearing in the poem. Then he calculated the pair combination of vowel and consonants and the results obviously showed that they are not independent. Now all he needed is to generate the initial consonant vowel distribution starting from the probabilities for the combination. We will simulate that with a sample text (UDHR) and show that law of large numbers can even appear with dependent events.

In [ ]:
RAW_TEXT = DATA_DIR.read_text(encoding="utf-8")
filtered_text = "".join(ch.lower() for ch in RAW_TEXT if ch.isalpha())

VOWELS = ["a","e","i","o","u"]

length = len(filtered_text)
vowels_cnt=0
consonant_cnt=0

for char in filtered_text:
    if char in VOWELS: vowels_cnt+=1
    else: consonant_cnt+=1

pct = [consonant_cnt/length,vowels_cnt/length]
pair_pct = [[0,0],[0,0]] ## CC CV VC VV

for i in range(1,length):
    prev_vowel = filtered_text[i-1] in VOWELS
    cur_vowel = filtered_text[i] in VOWELS
    
    pair_pct[prev_vowel][cur_vowel]+=1

for i in range(2):
    for j in range(2):
        pair_pct[i][j]/=length-1

# We have p(C,C) = P(C|C)P(C) => P(C|C) = P(C,C)/P(C)

graph = [[pair_pct[i][j]/pct[i] for j in range(2)] for i in range(2)]
print(graph)

rec.cache_clear()

for n in [10,100,200,1000,1500,2000]:
    print("When n is",n)
    res1 = rec(n,0,1)
    res2 = rec(n,0,0)
    res3 = rec(n,1,0)
    res4 = rec(n,1,1)
    
    print(res2,res3)
    print(res1,res4)
    print()

print(pct)

# There is a small difference that can be accounted due to the fact that we used n values for the percentage of consonant and vowels but the pairs are n-1 but we can ignore it 

From that humble start Markov Chain has contributed a lot to our society. It was used in monte carlo algorithm by Stanislaw Ulam and in page ranking system for page search system that was later named google. It simply relies on the hyperlinks a page has to other pages and uses that to make a probability and then walks randomly and the converged percentages will help rank these pages. And Since some pages might be in a loop the algorithm will spawn to other pages randomly to ensure all are visited. In addition to this the father of Information theory, Claude Shannon realized that he can go back to Markov's first experiment and instead of using the consonant and vowels he can use last few letters or even few words to make this prediction and it ended up being used for predictive text generation. We will experiment with those below.

In [ ]:
TEXT = "".join(text.lower() for text in RAW_TEXT)

class Tokenizer():
    def __init__(self,level:Literal["char","word"] = "char",num_prev:int=4,text:str=TEXT):
        self.level = level
        
        self.validate_input(num_prev)
        self.num_prev = num_prev
        self.train_text = text

    @property
    def result(self) -> Tuple[str,str]:
        if self.level=="char":
            return list(self.train_text), list("the human ")
        
        tokenized = re.findall(r"\S+",self.train_text)
        
        n = len(tokenized)
        idx = random.randint(0,(n-self.num_prev)//2) # Get random index around the center enough for num_prev
        
        return tokenized, tokenized[idx:idx+self.num_prev+1]

    def validate_input(self,num):
        assert 0<num<=10, "number of previous words or characters should be between 1 and 10"

class TextGenerator():
    def __init__(self, tokenizer:Tokenizer):
        self.tokenizer = tokenizer
        self.tokens,self.seed_text = self.tokenizer.result
        self.transitions = defaultdict(Counter)

        self.trained=False

    def train(self):
        if self.trained:
            return 
        
        n=self.tokenizer.num_prev    
        
        for i in range(n,len(self.tokens)):
            context = tuple(self.tokens[i-n:i])
            next_token = self.tokens[i]

            self.transitions[context][next_token]+=1

        self.trained=True

    def predict_next(self,context:Tuple):
        assert self.trained,"You need to train the text generator before attempting to use it"

        counts = self.transitions.get(context)
        if not counts: 
            return {}

        total = sum(counts.values())

        return {
            token: count/total
            for token,count in counts.most_common()
        }

    def generate(self,n:int):
        context = tuple(self.seed_text[:self.tokenizer.num_prev])
        output = list(context)

        for _ in range(n):
            pred = self.predict_next(context)

            if not pred:
                context = tuple(self.seed_text[:self.tokenizer.num_prev])
                continue 

            candidates = list(pred.keys())
            weights = list(pred.values())
            
            next_token = random.choices(candidates,weights=weights,k=1)[0]

            output.append(next_token)
            context = tuple(output[-self.tokenizer.num_prev:])

        return " ".join(output) if self.tokenizer.level=="word" else "".join(output)
        
        

for level in ["char","word"]:
    for num_prev in [1,3,4,7,10]:
        print(level,"based text generation by observing the last",num_prev,"words")
        print()
        
        tokenizer = Tokenizer(level,num_prev)
        generator = TextGenerator(tokenizer)

        generator.train()
        result = generator.generate(200) if level=="word" else generator.generate(600)
        print(result)

        print("="*80)

Markov chain is a memoryless system and while this is good in most cases it also becomes its downside. It doesn't account the momentum a system accumultated while reaching a certain point and this makes it inefficient for some problems especially those involving positive feedback loops. Markov chain can also be used to predict after how many actions things start to be fair or random. One example for this is shuffling cards and it can be used to predict that and we will see that below.